# IDTrack marketing overview — single multi-panel figure (cache-derived)

This notebook generates a **single, manuscript-style multi-panel overview** that can be used for:

- a Results “preview” figure (if you decide to include it)
- talk slides / lab presentations
- a reviewer response figure when you want to summarize the whole story quickly

Design constraints:

- **Cache-derived only**: no graph building is performed here.
- **Friction-free**: panels are rendered opportunistically; if an upstream notebook hasn't been run yet, that panel shows a placeholder.

## Outputs

- `idtrack-manuscript/figures/fig_idtrack_marketing_overview.pdf`

## Panel plan

- (a) Time-travel matrix (IDTrack) — “time is an axis”
- (b) External-mapper delta heatmap — “time travel recovers coverage”
- (c) Capability matrix — “reproducibility controls”
- (d) Stress-test entropy — “ambiguity is measurable, not hidden”


In [ ]:
from __future__ import annotations

from pathlib import Path

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt

try:
    import seaborn as sns
except Exception:  # noqa: S110
    sns = None

import sys

# Add experiments/src to sys.path (repo-relative; works from nested notebook dirs)
REPO_ROOT = Path.cwd().resolve()
while REPO_ROOT != REPO_ROOT.parent and not ((REPO_ROOT / 'idtrack').is_dir() and (REPO_ROOT / 'idtrack-manuscript').is_dir()):
    REPO_ROOT = REPO_ROOT.parent

EXPERIMENTS_SRC = REPO_ROOT / 'idtrack' / 'reproducibility' / 'experiments' / 'src'
sys.path.append(str(EXPERIMENTS_SRC))

from experiments_utils import MANUSCRIPT_COLORS, notebook_context, read_pickle, save_figure, label_panels  # noqa: E402
from plotting_utils import heatmap  # noqa: E402
from time_travel_analysis import add_fraction_columns, aggregate_bootstraps  # noqa: E402

ctx = notebook_context('marketing_overview', start=REPO_ROOT)
MANUSCRIPT_FIGURES = ctx.manuscript_figures
MANUSCRIPT_TABLES = ctx.manuscript_tables

print('Repo root:', REPO_ROOT)
print('IDTRACK_LOCAL_REPO:', ctx.idtrack_local_repo)
print('MANUSCRIPT_FIGURES:', MANUSCRIPT_FIGURES)
print('MANUSCRIPT_TABLES:', MANUSCRIPT_TABLES)


In [ ]:
# -------------------- Locate upstream artifacts (cache-first; no graph builds) --------------------

TT_DIR = Path(ctx.idtrack_local_repo) / 'experiments' / 'time_travel_matrix'
TT_CANDIDATES = list(TT_DIR.glob('time_travel_matrix_grid_*.pickle'))
TT_PKL = max(TT_CANDIDATES, key=lambda p: p.stat().st_mtime) if TT_CANDIDATES else None

DELTA_CSV = MANUSCRIPT_TABLES / 'time_travel_vs_external_mappers_deltas.csv'
ENTROPY_CSV = MANUSCRIPT_TABLES / 'random_stress_outcome_entropy.csv'
CAP_CSV = MANUSCRIPT_TABLES / 'tool_comparison_capability_matrix.csv'

print('TT_PKL:', TT_PKL if TT_PKL else '(missing)')
print('DELTA_CSV:', DELTA_CSV if DELTA_CSV.exists() else '(missing)')
print('ENTROPY_CSV:', ENTROPY_CSV if ENTROPY_CSV.exists() else '(missing)')
print('CAP_CSV:', CAP_CSV if CAP_CSV.exists() else '(missing)')


In [ ]:
# -------------------- Build the multi-panel overview figure --------------------

fig, axes = plt.subplots(2, 2, figsize=(14.5, 10.8), constrained_layout=True)
axA, axB, axC, axD = axes.ravel()

# (a) Time-travel matrix (IDTrack)
if TT_PKL is None:
    axA.axis('off')
    axA.text(0.5, 0.5, 'Missing time-travel matrix cache\nRun: experiment_time_travel_matrix/00_build_time_travel_matrix_cache.ipynb', ha='center', va='center')
else:
    grid = read_pickle(TT_PKL)
    df = add_fraction_columns(grid)
    agg = aggregate_bootstraps(df, group_cols=['from_release', 'to_release', 'final_database'])
    final_db = 'HGNC Symbol' if 'HGNC Symbol' in set(agg['final_database']) else 'Ensembl gene'
    sub = agg[agg['final_database'] == final_db]
    mat = sub.pivot(index='from_release', columns='to_release', values='frac_tdm_total').sort_index().sort_index(axis=1)
    heatmap(axA, mat, title=f'Time travel (IDTrack): TDM total\n{final_db}', cmap='viridis', vmin=0, vmax=1, square=True, cbar=True, cbar_label='fraction')
    axA.set_xlabel('to_release')
    axA.set_ylabel('from_release')

# (b) External-mapper delta heatmap (coverage gain)
if not DELTA_CSV.exists():
    axB.axis('off')
    axB.text(0.5, 0.5, 'Missing external delta table\nRun: experiment_time_travel_vs_external_mappers/01_analyze_time_travel_vs_external_mappers.ipynb', ha='center', va='center')
else:
    deltas = pd.read_csv(DELTA_CSV)
    # Prefer HGNC for a manuscript-oriented story
    target = 'HGNC Symbol' if 'HGNC Symbol' in set(deltas['target_db']) else str(deltas['target_db'].iloc[0])
    sub = deltas[deltas['target_db'] == target].copy()
    if sub.empty:
        axB.axis('off')
        axB.text(0.5, 0.5, f'No delta rows for target={target}', ha='center', va='center')
    else:
        sub['gain_coverage'] = -sub['delta_frac_1_to_0']
        ordered = sorted(set(sub['method'].astype(str)))
        mat = sub.pivot(index='method', columns='from_release', values='gain_coverage').reindex(ordered)
        vmax = float(np.nanmax(np.abs(mat.values))) if mat.size else 0.1
        vmax = max(vmax, 0.05)
        heatmap(
            axB,
            mat,
            title=f'External coverage gain after time travel\n(target={target})',
            cmap='RdBu_r',
            vmin=-vmax,
            vmax=vmax,
            center=0.0,
            square=False,
            cbar=True,
            cbar_label='gain',
        )
        axB.set_xlabel('from_release')
        axB.set_ylabel('method')

# (c) Capability matrix
if CAP_CSV.exists():
    cap = pd.read_csv(CAP_CSV)
    if 'Tool' in cap.columns:
        tools = cap['Tool'].astype(str).tolist()
        mat = cap.drop(columns=['Tool']).set_index(pd.Index(tools))
    else:
        mat = cap.set_index(cap.columns[0])
else:
    # Deterministic fallback (same as the Fig 4d notebook)
    capabilities = [
        'Pin historical Ensembl release',
        'Snapshot boundary (history window)',
        'Assembly-aware build axis',
        'Explainable / auditable paths',
        'External allowlist as contract',
        'Offline rerunnable after caching',
    ]
    tools = ['IDTrack', 'pybiomart', 'mygene', 'g:Profiler', 'gget']
    mat = pd.DataFrame(0, index=tools, columns=capabilities)
    mat.loc['IDTrack', :] = 1
    mat.loc['pybiomart', 'Pin historical Ensembl release'] = 1

heatmap(axC, mat.astype(float), title='Capability matrix (marketing, not accuracy)', cmap=['#FFFFFF', MANUSCRIPT_COLORS['1→1']], vmin=0, vmax=1, square=False, cbar=False)
axC.set_xlabel('')
axC.set_ylabel('')

# (d) Stress-test entropy scatter
if not ENTROPY_CSV.exists():
    axD.axis('off')
    axD.text(0.5, 0.5, 'Missing entropy table\nRun: experiment_random_stress_tests/00_random_stress_tests_fig4ab.ipynb', ha='center', va='center')
else:
    ent = pd.read_csv(ENTROPY_CSV)
    if {'frac_1_to_0', 'entropy', 'mode'}.issubset(ent.columns):
        for mode, marker in [('strict', 'o'), ('all', 's')]:
            sub = ent[ent['mode'].astype(str) == mode]
            if sub.empty:
                continue
            axD.scatter(sub['frac_1_to_0'], sub['entropy'], s=40, alpha=0.85, label=f'mode={mode}', marker=marker)
        axD.legend(frameon=True, fontsize=9)
        axD.set_xlim(0, 1)
        axD.set_xlabel('1→0 fraction')
        axD.set_ylabel('Outcome entropy')
        axD.set_title('Stress tests: compact ambiguity index (entropy)')
    else:
        axD.axis('off')
        axD.text(0.5, 0.5, 'Entropy CSV missing expected columns', ha='center', va='center')

label_panels(axes.ravel())

written = save_figure(fig, 'fig_idtrack_marketing_overview.pdf', ctx, formats=('pdf',))
print('Saved:', written['pdf'])
